On commence par un cas de classification tabulaire avec une regression logisitique sur 5 datasets de UCI.
Ensuite on s'intéresse à de la classification par Image et aux propriétées des classifieurs obtenus par Nll, PO(I) et PO(Beta^Star).

In [1]:
!git clone https://github.com/gabrielsinger2/nll_to_po.git

Cloning into 'nll_to_po'...
remote: Enumerating objects: 424, done.
remote: Counting objects: 100% (424/424), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 424 (delta 183), reused 384 (delta 152), pack-reused 0 (from 0)
Receiving objects: 100% (424/424), 13.80 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (183/183), done.


In [ ]:
%cd /content/nll_to_po
%pip install -e .

# Redémarre le runtime pour que l'import prenne bien:
import os, sys; os.kill(os.getpid(), 9)


/content/nll_to_po
Obtaining file:///content/nll_to_po
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
Reason for being yanked: Missing schema files
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.2/113.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.1/112.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.5 MB/s eta 0:00:00
  Building editable for NLL_TO_PO (pyproject.toml) ... done
  Created wheel for NLL_TO_PO: filename=nll_to_po-0.1.dev83+g13941a7b8-0.editable-py3-none-any.whl size=2956 sha256=edcb57b836cd2eca4b25ce79cd36ceab0b7ad4

In [13]:
# !pip install scikit-learn pandas matplotlib seaborn torch --quiet

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_wine, load_breast_cancer, load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

import nll_to_po.training.loss as L
import nll_to_po.training.reward as R
import nll_to_po.models.dn_policy as Policy
from nll_to_po.training.utils import train_single_policy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

sns.set_theme(style="whitegrid", font_scale=1.5)
sns.set_palette("colorblind")
sns.despine()

<Figure size 640x480 with 0 Axes>

In [14]:
def load_uci(
    dataset="wine", test_size=0.2, val_size=0.2, batch_size=256, standardize=True
):
    if dataset == "wine":
        data = load_wine()
    if dataset == "wine":
        data = load_wine()
    elif dataset == "iris":
        data = load_iris()
    elif dataset == "breast_cancer":
        data = load_breast_cancer()
    elif dataset == "load_digits":
        data = load_digits()
    else:
        raise ValueError(f"Unknown dataset: {dataset}")
    X, y = data.data, data.target
    if standardize:
        scaler = StandardScaler().fit(X)
        X = scaler.transform(X)

    # train / (val+test)
    X_tr, X_tt, y_tr, y_tt = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=0
    )
    # split val from train
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=0
    )
    # conversion en tensor type pour le donner au dataloader
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    X_tt = torch.tensor(X_tt, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_val = torch.tensor(y_val, dtype=torch.long)
    y_tt = torch.tensor(y_tt, dtype=torch.long)

    tr_loader = DataLoader(
        TensorDataset(X_tr, y_tr, y_tr, torch.zeros_like(y_tr)),
        batch_size=batch_size,
        shuffle=True,
    )
    # on ne shuffle pas la val et le test
    val_loader = DataLoader(
        TensorDataset(X_val, y_val, y_val, torch.zeros_like(y_val)),
        batch_size=batch_size,
        shuffle=False,
    )
    tst_loader = DataLoader(
        TensorDataset(X_tt, y_tt, y_tt, torch.zeros_like(y_tt)),
        batch_size=batch_size,
        shuffle=False,
    )

    meta = {"input_dim": X.shape[1], "num_classes": len(np.unique(y))}
    return tr_loader, val_loader, tst_loader, meta


train_loader, val_loader, test_loader, meta = load_uci("wine", batch_size=128)

Set up $\widehat{Y}$ c'est le one hot encoding des labels Y et je travaille avec les one hot encoding

In [15]:
import torch
import torch.nn.functional as F
from typing import Optional

@torch.no_grad()
def estimate_half_precision_onehot(
    train_loader,
    num_classes: int,
    device: str = "cpu",
    dtype: torch.dtype = torch.float64,
    ridge: Optional[float] = None,
):
    """
    Returns 0.5 * (Sigma + ridge * I)^{-1} for the empirical covariance of one-hot labels.
    Works with any DataLoader that yields (xb, yb, ...) with labels at index 1.

    Sigma = diag(p) - p p^T, where p is the empirical class frequency vector.

    Args
    ----
    train_loader : torch.utils.data.DataLoader
        Batches where labels are the second item.
    num_classes : int
        Number of classes C.
    device : str
        Device for the returned tensor ("cpu" or "cuda").
    dtype : torch.dtype
        Floating dtype (use float64 for numerical stability).
    ridge : Optional[float]
        Ridge parameter λ. If None, a small automatic λ is used:
        λ = 1e-6 * trace(Sigma) / C (and bumped up if Sigma is ill-conditioned).

    Returns
    -------
    torch.Tensor
        Tensor of shape (C, C): 0.5 * (Sigma + λ I)^{-1} in `dtype` on `device`.
    """
    # ---- 1) Accumulate counts without materializing N x C ----
    counts = torch.zeros(num_classes, dtype=torch.long)
    total = 0
    for batch in train_loader:
        yb = batch[1]
        if not torch.is_tensor(yb):
            yb = torch.as_tensor(yb)
        yb = yb.view(-1).to(torch.long)
        counts += torch.bincount(yb, minlength=num_classes).cpu()
        total += yb.numel()

    if total == 0:
        raise ValueError("Empty loader: no labels seen.")

    # Empirical class frequencies
    p = counts.to(dtype=dtype) / float(total)  # (C,)

    # ---- 2) Empirical covariance Sigma = diag(p) - p p^T ----
    Sigma = torch.diag(p) - torch.outer(p, p)  # (C, C)
    Sigma = Sigma.to(dtype=dtype)

    # ---- 3) Choose ridge if not provided ----
    C = num_classes
    if ridge is None:
        trace_S = torch.trace(Sigma)
        # Small scale-aware ridge; ensure strictly positive
        ridge = float((trace_S / max(C, 1)) * 1e-6)
        ridge = max(ridge, 1e-12)

    # ---- 4) Invert with ridge; try Cholesky first, fall back to solve/svd ----
    I = torch.eye(C, dtype=dtype)
    Sigma_reg = Sigma + ridge * I

    try:
        # Cholesky for speed/stability
        L = torch.linalg.cholesky(Sigma_reg)
        # Solve (Sigma_reg)^{-1} via two triangular solves
        inv_Sigma = torch.cholesky_inverse(L)
    except RuntimeError:
        # Fallback: symmetric solve; if still unstable, use SVD pseudoinverse with ridge
        try:
            inv_Sigma = torch.linalg.inv(Sigma_reg)
        except RuntimeError:
            U, S, Vh = torch.linalg.svd(Sigma, full_matrices=False)
            S_reg = S + ridge
            inv_Sigma = (Vh.T @ torch.diag(1.0 / S_reg) @ U.T)

    half_precision = 0.5 * inv_Sigma
    return half_precision.to(device=device)


In [16]:
@torch.no_grad()
def Estimate_trace_sigma_onehot(train_loader, num_classes: int) -> float:
    #modification pour qu'elle accetpe nimporte quel dataloader avec des labels en deuxieme position
    ys = []
    print("Estimating trace of label covariance matrix...")
    for _, yb, *rest in train_loader:
        ys.append(yb)
    print("Done")
    y = torch.cat(ys, dim=0)  # (N,)
    Y = F.one_hot(y, num_classes=num_classes).float()  # (N,C)
    Yc = Y - Y.mean(dim=0, keepdim=True)  # (N,C)
    Sigma = (Yc.T @ Yc) / max(Y.shape[0] - 1, 1)  # (C,C)
    return float(torch.trace(Sigma))

import torch

def _trace_from_counts(counts: torch.Tensor) -> float:
    """
    counts: 1D Long tensor of length C with class counts.
    """
    N = int(counts.sum().item())
    if N <= 1:
        return 0.0
    p2_sum = (counts.float() / N).pow(2).sum().item()
    return float((N / (N - 1)) * (1.0 - p2_sum))

@torch.no_grad()
def estimate_trace_sigma_onehot(train_loader, num_classes: int, device: str = "cpu") -> float:
    print('in estimate_trace_sigma_onehot')
    """
    Fast: streams the loader once, accumulates class counts via bincount.
    Works with any DataLoader where labels are at index 1.
    Accepts integer labels or one-hot labels (NxC).
    """
    counts = torch.zeros(num_classes, dtype=torch.long, device=device) # Create counts tensor on the specified device
    print("finished counts")
    print("len(dataset) =", len(train_loader.dataset))
    print("len(loader)  =", len(train_loader))  # nb de batches attendu

    for batch in train_loader:
        yb = batch[1].to(device) # Move yb to the specified device
        # If labels are one-hot, convert to class indices
        if yb.ndim == 2 and yb.size(1) == num_classes:
            yb = yb.argmax(dim=1)
        yb = yb.to(torch.long).flatten()
        counts += torch.bincount(yb, minlength=num_classes)
    return _trace_from_counts(counts)



def beta_star_from_data(train_loader, entropy_weight: float, num_classes: int, device: str = "cpu") -> float:
    print('in beta_star_from_data')
    tr = estimate_trace_sigma_onehot(train_loader, num_classes=num_classes, device=device) # Pass device to estimate_trace_sigma_onehot
    return float(entropy_weight * num_classes / (2.0 * max(tr, 1e-12)))

In [17]:
@torch.no_grad()
def evaluate_accuracy_multi_regression(
    policy: nn.Module, data_loader: DataLoader, device: str = "cuda"
) -> tuple[float, float]:
    policy.to(device)
    policy.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    for xb, yb, *rest in data_loader:
        xb, yb = xb.to(device), yb.to(device) # Move data to device
        _, prob_predit = policy(xb)

        # Top-1 accuracy
        y_prob_top1 = prob_predit.argmax(dim=-1)
        correct_top1 += (y_prob_top1 == yb).sum().item()

        # Top-5 accuracy
        _, top5_preds = prob_predit.topk(5, dim=-1)
        correct_top5 += torch.sum(top5_preds.eq(yb.view(-1, 1))).item()

        total += yb.numel()

    if total == 0:
        return 0.0, 0.0  # Handle empty dataloader case
    else:
        return correct_top1 / total, correct_top5 / total

In [18]:
import numpy as np
import tensorflow_datasets
from tensorflow_datasets.core.utils.lazy_imports_utils import pandas as pd
from tensorflow_datasets.core.utils.lazy_imports_utils import tensorflow as tf

#human_labels_np_path = '/content/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy'
#human_labels_csv_path = '/content/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human_annotations.csv'

#with tf.io.gfile.GFile(human_labels_np_path, "rb") as f:
#  human_annotations = np.load(f, allow_pickle=True)

#df = pd.DataFrame(human_annotations[()])

#with tf.io.gfile.GFile(human_labels_csv_path, "w") as f:
#  df.to_csv(f, index=False)


In [19]:
import numpy as np
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset
from torchvision import datasets, transforms
from PIL import Image
from typing import Optional, Dict

# CIFAR-10 normalization
_CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
_CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

class CIFAR10NTrain(Dataset):
    """
    Wraps torchvision CIFAR-10 (train=True) and replaces labels by a provided array (len=50000).
    Keeps the image order intact.
    """
    def __init__(self, root: str, noisy_labels: np.ndarray, transform=None, download: bool = True):
        base = datasets.CIFAR10(root=root, train=True, download=download)
        assert len(base.data) == 50000 and len(noisy_labels) == 50000
        self.data = base.data                      # N x 32 x 32 x 3 (uint8)
        self.labels = noisy_labels.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        img = Image.fromarray(self.data[idx])
        y = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        # Return placeholder tensors for mu and std to match the expected format
        return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

def _load_cifar10n_labels(
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
) -> Dict[str, np.ndarray]:
    """
    Load the CIFAR-10N dict from a .npy file (allow_pickle=True) or accept a preloaded dict.
    Returns a dict with keys like: 'clean_label', 'aggre_label', 'worse_label', 'random_label1/2/3', ...
    """
    if human_labels_dict is None:
        assert human_labels_np_path is not None, "Provide human_labels_np_path or human_labels_dict."
        d = np.load(human_labels_np_path, allow_pickle=True)
        # Some .npy store a 0-d object array containing the dict
        if isinstance(d, np.ndarray) and d.dtype == object:
            human_labels_dict = d.item()
        elif isinstance(d, dict):
            human_labels_dict = d
        else:
            raise ValueError("Unexpected CIFAR-10N npy format; expected dict or object array with dict.")
    return human_labels_dict

def load_cifar10n_dataset(
    batch_size: int = 128,
    data_root: str = "./data",
    label_key: str = "random_label1",
    val_fraction: float = 0.2,
    seed: int = 0,
    human_labels_np_path: Optional[str] = None,
    human_labels_dict: Optional[Dict[str, np.ndarray]] = None,
    download: bool = False,
):
    """
    Returns (train_loader, val_loader, test_loader, meta) using CIFAR-10 images and CIFAR-10N labels.

    - Train/Val split is performed on the 50k training images (shuffle once with `seed`).
    - Labels for BOTH train and val come from `label_key` (noisy validation, like your MNIST helper).
      If you want a clean-val variant, ask and I’ll add a switch.
    - Test set is the standard clean CIFAR-10 test.
    """
    # --- transforms ---
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])

    dl_kwargs = dict(num_workers=0, pin_memory=False, persistent_workers=False)

    if label_key == "clean_label":
        # Load standard CIFAR-10
        train_dataset_base = datasets.CIFAR10(root=data_root, train=True, download=download, transform=train_tf)
        test_dataset_base = datasets.CIFAR10(root=data_root, train=False, download=download, transform=eval_tf)

        # Create datasets with placeholder tensors
        class CIFAR10WithPlaceholders(Dataset):
            def __init__(self, base_dataset):
                self.base = base_dataset

            def __len__(self):
                return len(self.base)

            def __getitem__(self, idx):
                # Unpack all returned values and take the first two
                data = self.base[idx]
                img, y = data[0], data[1]
                return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

        train_val_dataset = CIFAR10WithPlaceholders(train_dataset_base)
        test_dataset = CIFAR10WithPlaceholders(test_dataset_base)

        # Split train -> train/val (80/20)
        num_train = len(train_val_dataset)
        indices = np.arange(num_train)
        split = int(val_fraction * num_train)
        rng = np.random.RandomState(seed)
        rng.shuffle(indices)
        val_idx, train_idx = indices[:split], indices[split:]

        train_loader = DataLoader(train_val_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx), **dl_kwargs)
        val_loader   = DataLoader(train_val_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx), **dl_kwargs)
        test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, **dl_kwargs)

    else:
        # Load CIFAR-10N label dict
        labels_dict = _load_cifar10n_labels(human_labels_np_path, human_labels_dict)

        # Support the common typo 'worst_label' -> 'worse_label'
        if label_key == "worst_label" and "worse_label" in labels_dict:
            label_key = "worse_label"

        assert label_key in labels_dict, f"{label_key=} not found. Available: {list(labels_dict.keys())}"
        noisy_labels = np.asarray(labels_dict[label_key])
        assert noisy_labels.shape == (50000,), "CIFAR-10N labels must be length 50000."

        # --- datasets ---
        # Modify to include placeholder tensors for mu and std
        train_val_dataset = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=train_tf, download=download)
        test_dataset_base = datasets.CIFAR10(root=data_root, train=False, download=download, transform=eval_tf)

        # Create a dataset for the test set that also includes placeholder tensors
        class CIFAR10TestWithPlaceholders(Dataset):
            def __init__(self, base_dataset):
                self.base = base_dataset

            def __len__(self):
                return len(self.base)

            def __getitem__(self, idx):
                # Unpack all returned values and take the first two
                data = self.base[idx]
                img, y = data[0], data[1]
                return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

        test_dataset = CIFAR10TestWithPlaceholders(test_dataset_base)


        # --- split 80/20 like your MNIST helper ---
        num_train = len(train_val_dataset)  # 50k
        indices = np.arange(num_train)
        split = int(val_fraction * num_train)
        rng = np.random.RandomState(seed)
        rng.shuffle(indices)
        val_idx, train_idx = indices[:split], indices[split:]

        train_loader = DataLoader(
            train_val_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx), **dl_kwargs
        )
        # Use eval transforms for val (no aug). Create a shallow copy with eval_tf.
        # Easiest: re-instantiate a view of the dataset but with eval_tf for the same labels.
        # Modify to include placeholder tensors for mu and std
        val_dataset_evalview_base = CIFAR10NTrain(root=data_root, noisy_labels=noisy_labels, transform=eval_tf, download=False)
        val_dataset_evalview = CIFAR10TestWithPlaceholders(val_dataset_evalview_base)

        val_loader = DataLoader(
            val_dataset_evalview, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx), **dl_kwargs
        )

        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **dl_kwargs)


    meta = {
        "in_channels": 3,
        "num_classes": 10,
        "image_size": (32, 32),
        "mean": _CIFAR10_MEAN,
        "std": _CIFAR10_STD,
        "label_key": label_key,
    }
    return train_loader, val_loader, test_loader, meta

In [20]:
import numpy as np
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler, Dataset
from torchvision import datasets, transforms

# CIFAR-100 normalization (commonly used stats)
_CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
_CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

class _WithPlaceholders(Dataset):
    """
    Wrap any torchvision dataset that returns (img, y) and append two placeholder tensors:
    returns (img, y, mu_placeholder, std_placeholder) to match your expected format.
    """
    def __init__(self, base_dataset: Dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx: int):
        img, y = self.base[idx]
        # placeholders as zeros (float32) shaped like scalar
        mu_ph  = torch.zeros((), dtype=torch.float32)
        std_ph = torch.zeros((), dtype=torch.float32)
        return img, int(y), mu_ph, std_ph


def load_cifar100_dataset(
    batch_size: int = 128,
    data_root: str = "./data",
    val_fraction: float = 0.2,
    seed: int = 0,
    download: bool = True,
):
    """
    Returns (train_loader, val_loader, test_loader, meta) for CIFAR-100 (clean labels only).

    - Train/Val split is performed on the 50k training images (shuffle once with `seed`).
    - Train uses standard aug (RandomCrop+Flip); Val/Test use eval transforms (no aug).
    - Each dataloader yields batches of (img, y, 0., 0.) to match your expected tuple format.
    """

    # --- transforms ---
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
    ])

    # Dataloader kwargs (tweak for your environment)
    dl_kwargs = dict(num_workers=0, pin_memory=False, persistent_workers=False)

    # --- base datasets ---
    train_base = datasets.CIFAR100(root=data_root, train=True,  download=download, transform=train_tf)
    val_view   = datasets.CIFAR100(root=data_root, train=True,  download=download, transform=eval_tf)   # same data, eval tf
    test_base  = datasets.CIFAR100(root=data_root, train=False, download=download, transform=eval_tf)

    # --- wrap to add placeholders ---
    train_wrapped = _WithPlaceholders(train_base)
    val_wrapped   = _WithPlaceholders(val_view)
    test_wrapped  = _WithPlaceholders(test_base)

    # --- split 80/20 (or val_fraction) on the 50k training set ---
    num_train = len(train_wrapped)  # 50_000
    indices = np.arange(num_train)
    split = int(val_fraction * num_train)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    val_idx, train_idx = indices[:split], indices[split:]

    # --- dataloaders ---
    train_loader = DataLoader(
        train_wrapped, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx), **dl_kwargs
    )
    val_loader = DataLoader(
        val_wrapped, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx), **dl_kwargs
    )
    test_loader = DataLoader(
        test_wrapped, batch_size=batch_size, shuffle=False, **dl_kwargs
    )

    meta = {
        "in_channels": 3,
        "num_classes": 100,
        "image_size": (32, 32),
        "mean": _CIFAR100_MEAN,
        "std": _CIFAR100_STD,
        "label_key": "clean_label",  # for compatibility with your other loaders
        "dataset_name": "CIFAR100",
    }

    return train_loader, val_loader, test_loader, meta


In [21]:
human_labels_np_path = "/content/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy"

#train_loader, val_loader, test_loader, meta = load_cifar10n_dataset(
#    batch_size=128,
#    data_root="./data",
#    label_key="random_label2",
#    val_fraction=0.2,
#    seed=0,
#    human_labels_np_path=human_labels_np_path,
#    download=True,
#)

#print(meta)

In [22]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, SubsetRandomSampler
import numpy as np
import torch

def load_mnist_dataset(batch_size: int = 128):
    transform = transforms.Compose([transforms.ToTensor()])  # (B,1,28,28) dans [0,1]
    train_dataset_base = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
    test_dataset_base  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    # Create datasets with placeholder tensors to match expected format
    class MNISTWithPlaceholders(Dataset):
        def __init__(self, base_dataset):
            self.base = base_dataset

        def __len__(self):
            return len(self.base)

        def __getitem__(self, idx):
            img, y = self.base[idx]
            # Return placeholder tensors for mu and std
            return img, y, torch.zeros_like(torch.tensor(y, dtype=torch.float32)), torch.zeros_like(torch.tensor(y, dtype=torch.float32))

    train_dataset = MNISTWithPlaceholders(train_dataset_base)
    test_dataset = MNISTWithPlaceholders(test_dataset_base)


    # Split train -> train/val (80/20)
    num_train = len(train_dataset)
    indices = np.arange(num_train)
    split = int(0.2 * num_train)
    rng = np.random.RandomState(0)
    rng.shuffle(indices)
    val_idx, train_idx = indices[:split], indices[split:]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
    val_loader   = DataLoader(train_dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    meta = {"in_channels": 1, "num_classes": 10, "image_size": (28, 28)}
    return train_loader, val_loader, test_loader, meta

In [23]:
#train_loader, val_loader, test_loader, meta=load_mnist_dataset(batch_size=256)

In [24]:
len(set([1,1,1,2]))

2

In [25]:
import torch
from nll_to_po.models.dn_policy import CNNClassifier

In [26]:
#model = CNNClassifier(in_channels=1, num_classes=10)
#x = torch.randn(128, 1, 28, 28)
#logits, probs = model(x)
#print(logits.shape, probs.shape)  # -> torch.Size([128, 10]) torch.Size([128, 10])



In [40]:
import os
def run_one_dataset(
    dataset_name: str,
    n_updates: int = 10,
    n_experiments: int = 5,
    batch_size: int = 128,
    learning_rate: float = 1e-4,
    entropy_weight: float = 1e-3,
    type_of_data : str = "tabular",
    corrupted_inputs: bool = False,
    save_dir: str = "./saved_models" # Add save_dir parameter
):
    # Ensure save directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Set device to CPU
    device = torch.device("cuda")

    if type_of_data=="tabular":
        print(f"Running on tabular dataset: {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_uci(
        dataset=dataset_name, batch_size=batch_size, standardize=True
        )
        C = meta["num_classes"]
        D_in = meta["input_dim"]
    elif type_of_data=="image" and dataset_name=="mnist":
        print(f"Running on {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_mnist_dataset(
            batch_size=batch_size
        )
        C = meta["num_classes"]
        D_in = meta["in_channels"]
    #NOT NOISY TO CORRECT later dont want to recode everything!
    elif type_of_data=="image" and dataset_name=="cifar10":
      print(f"Running on {dataset_name}")
      train_loader, val_loader, test_loader, meta = load_cifar10n_dataset(
    batch_size=128,
    data_root="./data",
    label_key="clean_label", # Use "clean_label" to load standard CIFAR10
    val_fraction=0.2,
    seed=0,
    download=True, # Set to True to download the dataset if not already present
)

      print("Standard CIFAR10 dataset loaded successfully!")
      C=meta["num_classes"]
      D_in=meta["in_channels"]
    elif type_of_data=="image" and dataset_name=="cifar10n":
        # Corrected path to the .npy file
        human_labels_np_path = "/content/nll_to_po/notebook/CIFAR_NOISY_DATA/CIFAR-10_human.npy"

        train_loader, val_loader, test_loader, meta = load_cifar10n_dataset(
            batch_size=128,
            data_root="./data",
            label_key="random_label2",
            val_fraction=0.2,
            seed=0,
            human_labels_np_path=human_labels_np_path,
        )
        C=meta["num_classes"]
        D_in=meta["in_channels"]
    elif type_of_data=="image" and dataset_name=="cifar100":
        print(f"Running on {dataset_name}")
        train_loader, val_loader, test_loader, meta = load_cifar100_dataset(
            batch_size=batch_size,
            download=True, # Set to True to download the dataset if not already present
        )
        C = meta["num_classes"]
        D_in = meta["in_channels"]


    print('finished to load datasets')
    beta_star = beta_star_from_data(
        train_loader, entropy_weight=entropy_weight, num_classes=C, device=device
    )
    print(f"Estimated beta_star: {beta_star:.4e}")
    # beta_list = [1, beta_star]
    beta_list = [beta_star]

    curves = []
    tests = []

    for rep in range(n_experiments):
        # policy = MLPClassifier(D_in, C)
        if type_of_data=="tabular":
            policy = Policy.MulticlassLogisticRegression(D_in, C)
        elif type_of_data=="image":
            policy = Policy.ResNet18Vanilla(num_classes=C, in_channels=3)
        loss_fn = L.NLL_Classification()
        print("start training NLL")
        trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
            policy=policy,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            loss_function=loss_fn,
            n_updates=n_updates,
            learning_rate=learning_rate,
            wandb_run=None,
            tensorboard_writer=None,
            logger=None,
            scheduler_patience=20, early_stopping_patience=100, device=device
        )

        # Save the trained NLL model
        model_save_path = os.path.join(save_dir, f"{dataset_name}_NLL_rep{rep+1}.pth")
        torch.save(trained_policy.state_dict(), model_save_path)
        print(f"Saved NLL model to {model_save_path}")


        df_tr = (
            pd.DataFrame(train_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_tr["split"] = "train"
        df_tr["method"] = "NLL"
        df_tr["beta"] = np.nan
        df_tr["rep"] = rep
        df_val = (
            pd.DataFrame(val_metrics).reset_index().rename(columns={"index": "epoch"})
        )
        df_val["split"] = "val"
        df_val["method"] = "NLL"
        df_val["beta"] = np.nan
        df_val["rep"] = rep
        curves += [df_tr, df_val]

        # test_acc = evaluate_accuracy(policy, test_loader)
        test_acc, top5 = evaluate_accuracy_multi_regression(trained_policy, test_loader, device=device)
        tests.append(
            {
                "dataset": dataset_name,
                "method": "NLL",
                "beta": np.nan,
                "rep": rep,
                "test_accuracy": test_acc,
                "top_5_accuracy": top5,
            }
        )

    for beta in beta_list:
        U = torch.eye(C) * float(beta)
        U=U.to(device)
        reward = R.OneHotMahalanobis(U, num_classes=C)  # reward
        print(f"Running Policy Optimization for {beta}")
        for rep in range(n_experiments):
            policy = Policy.ResNet18Vanilla(num_classes=C, in_channels=3)
            loss_fn = L.PO_Entropy_Classification(
                reward_fn=reward,
                n_generations=50,
                use_rsample=False,
                reward_transform="none",
                entropy_weight=entropy_weight,
            )

            beta_trained_policy, train_metrics, val_metrics, _, _ = train_single_policy(
                policy=policy,
                train_dataloader=train_loader,
                val_dataloader=val_loader,
                loss_function=loss_fn,
                n_updates=n_updates,
                learning_rate=learning_rate,
                wandb_run=None,
                tensorboard_writer=None,
                logger=None,
                early_stopping_patience=100, device=device
            )

            # Save the trained PO model
            beta_label = "beta_star" if abs(beta - beta_star) < 1e-12 else str(beta).replace('.', '_')
            model_save_path = os.path.join(save_dir, f"{dataset_name}_PO_Entropy_{beta_label}_rep{rep+1}.pth")
            torch.save(beta_trained_policy.state_dict(), model_save_path)
            print(f"Saved PO_Entropy model to {model_save_path}")


            df_tr = (
                pd.DataFrame(train_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_tr["split"] = "train"
            df_tr["method"] = "PO_Entropy"
            df_tr["beta"] = beta
            df_tr["rep"] = rep
            df_val = (
                pd.DataFrame(val_metrics)
                .reset_index()
                .rename(columns={"index": "epoch"})
            )
            df_val["split"] = "val"
            df_val["method"] = "PO_Entropy"
            df_val["beta"] = beta
            df_val["rep"] = rep
            curves += [df_tr, df_val]

            # test_acc = evaluate_accuracy(policy, test_loader)
            test_acc, top5 = evaluate_accuracy_multi_regression(
                beta_trained_policy, test_loader, device=device
            )
            tests.append(
                {
                    "dataset": dataset_name,
                    "method": "PO_Entropy",
                    "beta": beta,
                    "rep": rep,
                    "test_accuracy": test_acc,
                    "top_5_accuracy": top5,
                }
            )

    curves_df = pd.concat(curves, ignore_index=True)
    tests_df = pd.DataFrame(tests)
    curves_df["is_beta_star"] = curves_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    tests_df["is_beta_star"] = tests_df["beta"].apply(
        lambda b: isinstance(b, float) and abs(b - beta_star) < 1e-12
    )
    return curves_df, tests_df, beta_star

In [37]:
import pandas as pd
def plot_curves_for_dataset(
    curves_df: pd.DataFrame, dataset_name: str, beta_star: float
):
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

    # helpers
    def make_label(row):
        if row["method"] == "NLL":
            return "NLL"
        if pd.isna(row["beta"]):
            return "PO (β=NA)"
        if abs(row["beta"] - beta_star) < 1e-12:
            return "PO (β*)"
        return f"PO (β={row['beta']:.3g})"

    curves_df = curves_df.copy()
    curves_df["label"] = curves_df.apply(make_label, axis=1)

    # colors: NLL blue, β* black, others red
    palette_map = {}
    for lab in curves_df["label"].unique():
        if lab == "NLL":
            palette_map[lab] = "red"
        elif lab == "PO (β*)":
            palette_map[lab] = "yellow"
        else:
            palette_map[lab] = "#1f77b4"

    # TRAIN
    sub = curves_df[curves_df["split"] == "train"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[0],
        palette=palette_map,
        legend=False,
    )
    ax[0].set_title(f"{dataset_name}: Train accuracy vs epoch")
    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("accuracy")

    # VAL
    sub = curves_df[curves_df["split"] == "val"]
    sns.lineplot(
        data=sub,
        x="epoch",
        y="accuracy",
        hue="label",
        errorbar=("ci", 95),
        ax=ax[1],
        palette=palette_map,
        legend=True,
    )
    ax[1].set_title(f"{dataset_name}: Val accuracy vs epoch")
    ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("accuracy")
    ax[1].legend(title="method", frameon=False, loc="lower right")

    plt.tight_layout()
    plt.show()

In [34]:
#datasets = ["iris"]

#all_curves = []
#all_tests = []

#for ds in datasets:
#    curves_df, tests_df, bstar = run_one_dataset(
#        dataset_name=ds,
#        n_updates=100,
#        n_experiments=100,
#        batch_size=128,
#        learning_rate=1e-2,
#        entropy_weight=1e-1,
#        type_of_data= "tabular"
#    )
#    plot_curves_for_dataset(curves_df, ds, bstar)
#    show_test_table(tests_df, ds)

#    curves_df["dataset"] = ds
#    tests_df["dataset"] = ds
#    all_curves.append(curves_df)
#    all_tests.append(tests_df)

#df_curves_all = pd.concat(all_curves, ignore_index=True)
#df_tests_all = pd.concat(all_tests, ignore_index=True)

In [38]:
def show_test_table(tests_df: pd.DataFrame, dataset_name: str):
    keep = (tests_df["method"] == "PO_Entropy") | (tests_df["method"] == "NLL")
    df = tests_df[keep].copy()

    grouped = (
        df.groupby(["dataset", "method", "is_beta_star"], dropna=False)["test_accuracy"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    def beta_label(row):
        if row["method"] == "NLL":
            return "—"
        return "β*" if row["is_beta_star"] else "β=1"

    grouped["beta_label"] = grouped.apply(beta_label, axis=1)
    grouped = grouped[["dataset", "method", "beta_label", "mean", "std", "count"]]
    grouped = grouped[grouped["dataset"] == dataset_name]

    print(f"Test accuracy summary — {dataset_name}")
    display(grouped.style.format({"mean": "{:.4f}", "std": "{:.4f}"}))

In [41]:
Datasets = ["cifar10"]

all_curves = []
all_tests = []

for ds in Datasets:
    curves_df, tests_df, bstar = run_one_dataset(
        dataset_name=ds,
        n_updates=50,
        n_experiments=1,
        batch_size=4*256,
        learning_rate=5*1e-2,
        entropy_weight=1, type_of_data="image"
    )
    plot_curves_for_dataset(curves_df, ds, bstar)
    show_test_table(tests_df, ds)

    curves_df["dataset"] = ds
    tests_df["dataset"] = ds
    all_curves.append(curves_df)
    all_tests.append(tests_df)

df_curves_all = pd.concat(all_curves, ignore_index=True)
df_tests_all = pd.concat(all_tests, ignore_index=True)

Running on cifar10


100%|██████████| 170M/170M [04:32<00:00, 625kB/s]


Standard CIFAR10 dataset loaded successfully!
finished to load datasets
in beta_star_from_data
in estimate_trace_sigma_onehot
finished counts
len(dataset) = 50000
len(loader)  = 313
Estimated beta_star: 5.5554e+00
start training NLL


Training epochs: 100%|██████████| 1/1 [00:24<00:00, 24.23s/it]


Saved NLL model to ./saved_models/cifar10_NLL_rep1.pth
Running Policy Optimization for 5.5554422924621685


Training epochs:   0%|          | 0/1 [00:07<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
df_curves_all.to_csv("2_50_lr_0_01_lambda01_CIFAR_df_curves_all.csv")
df_tests_all.to_csv("CIFAR_df_tests_all.csv")